In [12]:
# This must follow the Extract, Transform, Load (ETL) pattern.

import yaml
from treelib import Tree

import pandas as pd

from pathlib import Path
import os

from units_llnl import Measurement

# Extract
load experiment.yml
load grid files from grid nodes
extract node list from grid file

In [13]:
with open(Path("../config/MV-LV.yml"), "r") as f:
    cfg = yaml.safe_load(f)

In [14]:
tree = Tree()

def add_to_tree(tree, node_dict, parent=None):
    """Recursively add nodes from config dict to tree structure."""
    node_id = f"{parent}/{node_dict.get("id")}" if parent else node_dict.get("id")
    node_tag = node_dict.get("name")
    node_config = node_dict.get("config", {}).copy()
    node_config["type"] = node_dict.get("type")
    # Convert empty string values in node_config to None
    node_config = {k: (v if v != "" else None) for k, v in node_config.items()}
    
    # Create node with config data
    tree.create_node(
        tag=f"{node_tag}", 
        identifier=node_id, 
        parent=parent,
        data=node_config
    )
    
    for sub in node_dict.get("sub_federates", []):
        add_to_tree(tree, sub, parent=node_id)

add_to_tree(tree, cfg["federation"])
tree.show(idhidden=False)

MediumVoltageGrid[mv-grid_0]
├── LowVoltageGrid[mv-grid_0/lv-grid_0]
│   ├── Load profile house[mv-grid_0/lv-grid_0/loadhouse_0]
│   └── sim house[mv-grid_0/lv-grid_0/house_0]
│       ├── BYD Battery[mv-grid_0/lv-grid_0/house_0/battery_0]
│       ├── Smart HEMS[mv-grid_0/lv-grid_0/house_0/hems_0]
│       └── Sunwell PV[mv-grid_0/lv-grid_0/house_0/pv_1]
└── PV Power Plant SWM[mv-grid_0/pv_0]



In [15]:
# Extract grid nodes
grid_ids = [node_id for node_id in tree.expand_tree(filter=lambda x: x.data["type"] == "grid")]

# Find nodes in grid file or infDB
grid_nodes = {}
for grid_id in grid_ids:
    grid_node = tree.get_node(grid_id)
    layout = grid_node.data.get("layout")
    location = grid_node.data.get("location")
    
    if layout is None:
        if location is None:
            raise ValueError(f"No layout or location specified for grid {grid_id}")
        else:
            # TODO: infDB.load(location)
            grid_nodes[grid_id] = None
            print(f"Loading layout from infDB for {grid_id} at location {location}")

    else:        
        if location is not None:
            raise ValueError(f"Both layout and location specified for grid {grid_id}. Please specify only one.")
        else:
            layout_path = Path("../data/input") / layout
            grid_nodes[grid_id] = pd.read_excel(layout_path, sheet_name="load")["bus"].tolist()
            print(f"Loaded layout from file for {grid_id} from {layout_path}")

print(grid_nodes)

Loaded layout from file for mv-grid_0 from ../data/input/kerber_landnetz_freileitung_1.xlsx
Loaded layout from file for mv-grid_0/lv-grid_0 from ../data/input/kerber_landnetz_freileitung_1.xlsx
{'mv-grid_0': [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14], 'mv-grid_0/lv-grid_0': [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]}


# Transform

node mapping:
    extend grid node sub_federates according to pandapower grid files
    turn placement config for each sub_federate into sub_federates

validate node/sub_federate placement:
    check if node has multiple sub_federates
    check if sub_federates are outside of grid node list
    check if nodes are empty

turn into cst federate config

In [16]:
validation_errors = []

for node in tree.all_nodes():
    data = node.data
    # Determine type, default to None if missing
    node_type = data.get("type")
    
    if not node_type:
        validation_errors.append(f"[Structure] Node '{node.tag}' ({node.identifier}) is missing a 'type' definition.")
        continue

    match node_type:
        case "grid":
            layout = data.get("layout")
            location = data.get("location")
            if not layout and not location:
                validation_errors.append(f"[Grid] Node '{node.tag}' ({node.identifier}) must specify either 'layout' or 'location'.")
            if layout and location:
                validation_errors.append(f"[Grid] Node '{node.tag}' ({node.identifier}) specifies both 'layout' and 'location'.")

        case "load":
            # Check if at least one profile is assigned
            if not data.get("electrical_load") and not data.get("heat_load"):
                validation_errors.append(f"[Load] Node '{node.tag}' ({node.identifier}) requires 'electrical_load' or 'heat_load'.")

        case "house":
            if not data.get("model"):
                validation_errors.append(f"[House] Node '{node.tag}' ({node.identifier}) requires a 'model' definition.")
            # Placeholder for RL specific checks (e.g. agent config)
        
        case "pv":
             # PV nodes require generation profile data (seen in context variables as 'data')
            if not data.get("max_production"):
                validation_errors.append(f"[PV] Node '{node.tag}' ({node.identifier}) requires 'max_production' (profile path).")
            if not data.get("capacity"):
                validation_errors.append(f"[PV] Node '{node.tag}' ({node.identifier}) requires 'capacity' definition.")

        case "battery":
            if not data.get("capacity"):
                validation_errors.append(f"[Battery] Node '{node.tag}' ({node.identifier}) requires 'capacity' definition.")
            if not data.get("power"):
                validation_errors.append(f"[Battery] Node '{node.tag}' ({node.identifier}) requires 'power' definition.")
        
        case "hems":
            if not data.get("control_strategy"):
                validation_errors.append(f"[HEMS] Node '{node.tag}' ({node.identifier}) requires 'control_strategy' definition.")

        case _:
            # Strict validation: Unknown types are considered errors
            validation_errors.append(f"[Type] Unknown node type '{node_type}' at node '{node.tag}' ({node.identifier})")

if validation_errors:
    print("Configuration Invalid:")
    for error in validation_errors:
        print(f" - {error}")
    raise ValueError("Configuration validation failed due to errors listed above.")
else:
    print("Configuration successfully validated.")

Configuration successfully validated.


In [17]:
import copy
from treelib import Tree

def expand_grid_nodes(tree: Tree, grid_nodes: dict) -> Tree:
    """
    Expands the tree by placing federates onto grid buses according to 'placement' rules.
    - Removes original definition nodes from the grid parent.
    - Replicates subtrees for 'list' or 'fill' placements.
    - Ensures unique IDs for all placed nodes.
    """
    
    for grid_id, buses in grid_nodes.items():
        if not tree.contains(grid_id):
            continue
            
        # If no buses are defined for this grid, skip
        if not buses:
            continue

        # 1. Classify logic and extract prototypes
        # We process current children to build placement plan
        explicit_placements = {} # Map[bus_id] -> template_subtree
        fill_templates = []      # List[template_subtree]
        
        # Get current children nodes to iterate over
        children = tree.children(grid_id)
        
        for child in children:
            placement = child.data.get("placement")
            
            # Extract the full subtree (template) for this federate
            # We use remove_subtree to detach it from the main tree immediately
            # giving us a clean slate to paste back onto.
            template_subtree = tree.remove_subtree(child.identifier)
            
            if isinstance(placement, int):
                placement = [placement]  # Normalize to list for uniform processing

            if isinstance(placement, list):
                for p in placement:
                    if p in explicit_placements:
                         raise ValueError(f"Configuration Error: Bus {p} in grid '{grid_id}' is claimed by multiple federates.")
                    if p not in buses:
                        raise ValueError(f"Configuration Error: Federate '{child.tag}' placed on bus {p} which does not exist in grid '{grid_id}'.")
                    # We store the same template reference; we must deepcopy when pasting
                    explicit_placements[p] = template_subtree

            elif placement == "fill":
                fill_templates.append(template_subtree)
                
            else:
                # If placement is None/Empty in config, we treat it simply as not having a specific spot.
                # Since we stripped the tree, we discard it unless specific logic is needed.
                raise ValueError(f"Configuration Error: Federate '{child.tag}' does not have a valid placement in grid '{grid_id}'.")

        # 2. Re-populate the grid node with concrete instances per bus
        for bus in buses:
            # Determine which template to use
            template = None
            is_fill = False

            if bus in explicit_placements:
                template = explicit_placements[bus]
            elif fill_templates:
                # Use the first fill template available (simple logic)
                template = fill_templates[0]
                is_fill = True
            
            new_root_id = f"{grid_id}/bus_{bus}"

            if template:
                # We must modify the subtree to have unique IDs before pasting
                # Deepcopy ensures we don't mutate the template for other buses
                subtree_to_paste = copy.deepcopy(template)
                
                # Update IDs inside the subtree to be unique
                # Old Root ID -> New Root ID
                old_root_id = subtree_to_paste.root
                root_node = subtree_to_paste[old_root_id]
                
                # We need to systematically rename everything in this subtree
                # Mapping: old_id -> new_id
                # Strategy: Append grid and bus info to ensure global uniqueness
                
                # Logic: Rename the root specifically to the requested format
                # Rename descendants to strictly unique IDs
                
                # 1. Update root
                root_node.identifier = new_root_id
                # Update data to reflect actual placement
                root_node.data["placement"] = bus 
                subtree_to_paste.update_node(old_root_id, identifier=new_root_id)
                
                # 2. Update all other nodes in subtree
                # BFS/DFS transversal to rename. Note: changing IDs while iterating needs care.
                # treelib doesn't support bulk re-id easily, so we iterate keys
                for node_id in list(subtree_to_paste.nodes.keys()):
                    if node_id == new_root_id: 
                        continue # Already handled root
                    
                    # Generate unique ID by replacing the old root prefix with the new root ID
                    if node_id.startswith(old_root_id):
                        new_sub_id = node_id.replace(old_root_id, new_root_id, 1)
                    else:
                        new_sub_id = f"{new_root_id}/{node_id}"
                        
                    subtree_to_paste.update_node(node_id, identifier=new_sub_id)

                # Paste the prepared subtree back into the main tree
                tree.paste(grid_id, subtree_to_paste)

            else:
                # "if placement is empty or none make a node of type 'empty'"
                # (And no fill template was available)
                tree.create_node(
                    tag=f"Empty Slot {bus}",
                    identifier=new_root_id,
                    parent=grid_id,
                    data={"type": "empty", "bus": bus, "placement": bus}
                )

    return tree

# Execute the expansion
expand_grid_nodes(tree, grid_nodes)
tree.show(idhidden=False)

MediumVoltageGrid[mv-grid_0]
├── Empty Slot 10[mv-grid_0/bus_10]
├── Empty Slot 11[mv-grid_0/bus_11]
├── Empty Slot 12[mv-grid_0/bus_12]
├── Empty Slot 13[mv-grid_0/bus_13]
├── Empty Slot 14[mv-grid_0/bus_14]
├── Empty Slot 2[mv-grid_0/bus_2]
├── Empty Slot 4[mv-grid_0/bus_4]
├── Empty Slot 6[mv-grid_0/bus_6]
├── Empty Slot 7[mv-grid_0/bus_7]
├── Empty Slot 8[mv-grid_0/bus_8]
├── Empty Slot 9[mv-grid_0/bus_9]
├── LowVoltageGrid[mv-grid_0/bus_3]
│   ├── Load profile house[mv-grid_0/bus_3/loadhouse_0]
│   └── sim house[mv-grid_0/bus_3/house_0]
│       ├── BYD Battery[mv-grid_0/bus_3/house_0/battery_0]
│       ├── Smart HEMS[mv-grid_0/bus_3/house_0/hems_0]
│       └── Sunwell PV[mv-grid_0/bus_3/house_0/pv_1]
└── PV Power Plant SWM[mv-grid_0/bus_5]



## General

In [18]:
import pandas as pd

general_cfg = cfg.get("general", {})
errors = []

# Validate required fields
required_fields = ["end_time"]  # start_time can be defaulted
for field in required_fields:
    if field not in general_cfg or general_cfg[field] is None:
        errors.append(f"[General] Missing required field '{field}'.")

# Impute defaults
if "start_time" not in general_cfg or general_cfg["start_time"] is None:
    print("[General] 'start_time' missing, defaulting to '2023-01-01T00:00:00'.")
    general_cfg["start_time"] = "2023-01-01T00:00:00"

# Calculate timestamps if numeric duration is provided
start_time = general_cfg["start_time"]
end_time = general_cfg["end_time"]

try:
    # Ensure start_time is a valid timestamp string
    start_ts = pd.Timestamp(start_time)
    
    # Handle end_time: if it looks like a number (duration in seconds), calculate absolute time
    if isinstance(end_time, (int, float)):
        print(f"[General] 'end_time' is a duration ({end_time}s). Calculating absolute timestamp.")
        end_ts = start_ts + pd.Timedelta(seconds=end_time)
        general_cfg["end_time"] = end_ts.isoformat()
    elif isinstance(end_time, str):
        # Validate it parses correctly
        pd.Timestamp(end_time)
    else:
        errors.append(f"[General] 'end_time' format not recognized: {end_time}")

except Exception as e:
    errors.append(f"[General] Timestamp parsing error: {str(e)}")

# Raise errors if any found
if errors:
    print("General Configuration Invalid:")
    for error in errors:
        print(f" - {error}")
    raise ValueError("General configuration validation failed.")
else:
    print("General configuration successfully validated and processed.")
    if not general_cfg.get("name"):
        raise ValueError("[General] Missing required field 'name'.")

    print(general_cfg)

[General] 'start_time' missing, defaulting to '2023-01-01T00:00:00'.
[General] 'end_time' is a duration (86400s). Calculating absolute timestamp.
General configuration successfully validated and processed.
{'name': 'Demo Experiment Medium Voltage Grid with Houses', 'start_time': '2023-01-01T00:00:00', 'end_time': '2023-01-02T00:00:00', 'time_step': 300, 'duration': 86400}


add publications and subscriptions with their units 

In [19]:
def add_pub_sub(node, topic, unit, type="publication"):
    """Helper to add pub/sub to node data structure."""
    key = "publications" if type == "publication" else "subscriptions"
    if key not in node.data:
        node.data[key] = {}
    node.data[key][topic] = unit

# Iterate over all nodes to establish relationships based on parent-child structure
for parent_node in tree.all_nodes():
    parent_type = parent_node.data.get("type")
    children = tree.children(parent_node.identifier)
    
    for child_node in children:
        child_type = child_node.data.get("type")
        
        # --- Logic for Grid Parent ---
        if parent_type == "grid":
            # Grid publishes Voltage to children
            # Logic: Voltage is specific to the connection point, governed by Grid
            voltage_topic = f"{child_node.identifier}/voltage"
            add_pub_sub(parent_node, voltage_topic, "V", "publication")
            add_pub_sub(child_node, voltage_topic, "V", "subscription")

            # Grid subscribes to Power from children (House, Load, Battery, PV, Sub-Grid)
            if child_type in ["house", "load", "battery", "pv", "grid"]:
                p_topic = f"{child_node.identifier}/active_power"
                q_topic = f"{child_node.identifier}/reactive_power"
                
                # Parent (Grid) Subscribes
                add_pub_sub(parent_node, p_topic, "W", "subscription")
                add_pub_sub(parent_node, q_topic, "VAr", "subscription")
                
                # Child Publishes
                add_pub_sub(child_node, p_topic, "W", "publication")
                add_pub_sub(child_node, q_topic, "VAr", "publication")

            # Grid publishes Control JSON specifically to House
            if child_type == "house":
                control_topic = f"{child_node.identifier}/control"
                add_pub_sub(parent_node, control_topic, "json", "publication")
                add_pub_sub(child_node, control_topic, "json", "subscription")

        # --- Logic for House Parent ---
        elif parent_type == "house":
            # House acts as a local controller for HEMS, PV, Battery
            if child_type in ["pv", "battery", "hems"]:
                # House Subscribes to P, Q, V from components
                p_topic = f"{child_node.identifier}/active_power"
                q_topic = f"{child_node.identifier}/reactive_power"
                v_topic = f"{child_node.identifier}/voltage"
                
                # Parent (House) Subscribes
                add_pub_sub(parent_node, p_topic, "W", "subscription")
                add_pub_sub(parent_node, q_topic, "VAr", "subscription")
                add_pub_sub(parent_node, v_topic, "V", "subscription")
                
                # Child Publishes
                add_pub_sub(child_node, p_topic, "W", "publication")
                add_pub_sub(child_node, q_topic, "VAr", "publication")
                add_pub_sub(child_node, v_topic, "V", "publication")

                # House Publishes Control to components
                control_topic = f"{child_node.identifier}/control"
                add_pub_sub(parent_node, control_topic, "json", "publication")
                add_pub_sub(child_node, control_topic, "json", "subscription")

# Display a sample of the configuration to verify
print("Sample Node Configuration (mv-grid_0):")
print(tree.get_node("mv-grid_0").data.get("publications"))
print(tree.get_node("mv-grid_0").data.get("subscriptions"))

# If a House exists, check it too
try:
    # Find a house node ID
    house_node = [n for n in tree.all_nodes() if n.data.get("type") == "house"][0]
    print(f"\nSample Node Configuration ({house_node.tag}):")
    print(house_node.data.get("publications"))
    print(house_node.data.get("subscriptions"))
except IndexError:
    pass



Sample Node Configuration (mv-grid_0):
{'mv-grid_0/bus_2/voltage': 'V', 'mv-grid_0/bus_3/voltage': 'V', 'mv-grid_0/bus_4/voltage': 'V', 'mv-grid_0/bus_5/voltage': 'V', 'mv-grid_0/bus_6/voltage': 'V', 'mv-grid_0/bus_7/voltage': 'V', 'mv-grid_0/bus_8/voltage': 'V', 'mv-grid_0/bus_9/voltage': 'V', 'mv-grid_0/bus_10/voltage': 'V', 'mv-grid_0/bus_11/voltage': 'V', 'mv-grid_0/bus_12/voltage': 'V', 'mv-grid_0/bus_13/voltage': 'V', 'mv-grid_0/bus_14/voltage': 'V'}
{'mv-grid_0/bus_3/active_power': 'W', 'mv-grid_0/bus_3/reactive_power': 'VAr', 'mv-grid_0/bus_5/active_power': 'W', 'mv-grid_0/bus_5/reactive_power': 'VAr'}

Sample Node Configuration (sim house):
{'mv-grid_0/bus_3/house_0/active_power': 'W', 'mv-grid_0/bus_3/house_0/reactive_power': 'VAr', 'mv-grid_0/bus_3/house_0/battery_0/control': 'json', 'mv-grid_0/bus_3/house_0/pv_1/control': 'json', 'mv-grid_0/bus_3/house_0/hems_0/control': 'json'}
{'mv-grid_0/bus_3/house_0/voltage': 'V', 'mv-grid_0/bus_3/house_0/control': 'json', 'mv-grid_0/b

# Load

send to file or database

In [20]:
import json

runner_config = {
    "analysis": general_cfg.get("name", "GridLockAnalysis"),
    "federation": general_cfg.get("name", "GridLockFederation"),
    "start_time": general_cfg.get("start_time"),
    "stop_time": general_cfg.get("end_time"),
    "docker": True
}
print(json.dumps(runner_config, indent=4))

{
    "analysis": "Demo Experiment Medium Voltage Grid with Houses",
    "federation": "Demo Experiment Medium Voltage Grid with Houses",
    "start_time": "2023-01-01T00:00:00",
    "stop_time": "2023-01-02T00:00:00",
    "docker": true
}


In [21]:
from cosim_toolbox.sims import FederationConfig, FederateConfig, DockerRunner, Collect

# Define mapping helper
def map_params_to_type(federate_type):
    """Maps internal federate types to Docker images and commands."""
    mapping = {
        "grid": {
            "image": "gridlock-grid:latest",
            "command": "python3 main.py",
        },
        "house": {
            "image": "gridlock-house:latest",
            "command": "python3 main.py",
        },
        "recorder": {
            "image": "gridlock-recorder:latest",
            "command": "helics_recorder", 
        },
        "pv": {
             "image": "gridlock-house:latest", 
             "command": "python3 main.py"
        },
        "battery": {
             "image": "gridlock-house:latest",
             "command": "python3 main.py"
        },
        "hems": {
             "image": "gridlock-house:latest",
             "command": "python3 main.py"
        }
    }
    return mapping.get(federate_type, {
        "image": "cosim-cst:latest",
        "command": "python3 main.py"
    })

# 1. Setup Federation
name = general_cfg.get("name", "GridLock")
federation = FederationConfig(
    f"{name}Scenario",
    f"{name}Analysis",
    f"{name}Federation",
    True,
    "json",
    "csv"
)

# 2. Iterate nodes and create federates
for node in tree.all_nodes():
    data = node.data
    node_type = data.get("type")
    
    # Skip non-executable nodes
    if not node_type or node_type == "empty":
        continue

    # Create config
    time_step = general_cfg.get("time_step", 1.0)
    fed = FederateConfig(node.identifier, period=time_step)
    
    # Add config to federation
    federation.add_federate_config(fed)
    
    # Configure Docker/Process
    mapped = map_params_to_type(node_type)
    fed.config("image", mapped["image"])
    fed.config("command", mapped["command"])
    fed.config("federate_type", node_type) 
    
    # 3. Add Pubs/Subs directly
    for topic, unit in data.get("publications", {}).items():
        dtype = "string" if unit == "json" else "double"
        if not hasattr(fed, 'publications'):
             fed.publications = []
        fed.publications.append({
            "key": topic,
            "type": dtype,
            "unit": unit,
            "global": True
        })
        
    for topic, unit in data.get("subscriptions", {}).items():
        dtype = "string" if unit == "json" else "double"
        if not hasattr(fed, 'subscriptions'):
             fed.subscriptions = []
        fed.subscriptions.append({
            "key": topic,
            "type": dtype,
            "unit": unit,
            "required": True 
        })

# 4. Write Configuration Output
start_str = general_cfg["start_time"]
end_str = general_cfg["end_time"]

print("Generating configuration...")

try:
    federation.write_config(start_str, end_str)
    
    # 5. Generate Docker Compose
    DockerRunner.define_yaml(federation.scenario_name, use_meta_db="json")
    print("Federation configuration and docker-compose.yml successfully generated.")
except Exception as e:
    print(f"Error generating config: {e}")
    import traceback
    traceback.print_exc()

Generating configuration...
Writing configuration files to 'meta_store'...
Configuration files written successfully.
Federation configuration and docker-compose.yml successfully generated.


In [22]:
def map_params_to_type(federate_type):
    """Maps internal federate types to Docker images and commands."""
    mapping = {
        "grid": {
            "image": "gridlock-grid:latest",
            "command": "python3 main.py",
        },
        "house": {
             # "rl_house" or "csv_house" depending on config, but distinct types in tree is better.
             # Assuming RL house based on descriptions
            "image": "gridlock-house:latest",
            "command": "python3 main.py",
        },
        "recorder": {
            "image": "gridlock-recorder:latest",
            "command": "helics_recorder", 
        }
        # Add other types as necessary
    }
    return mapping.get(federate_type, {
        "image": "cosim-cst:latest",
        "command": "python3 main.py"
    })

def generate_federate_config(node, global_time_step):
    """Generates the configuration dictionary for a single federate."""
    node_data = node.data
    federate_id = node.identifier
    # Use tag as name if available, else identifier
    federate_name = node.tag if node.tag else federate_id
    federate_type = node_data.get("type", "unknown")
    
    # Skip empty slots or purely structural nodes if they aren't federates
    # However, in this tree structure, most nodes seem to imply a simulation entity unless "empty".
    if federate_type == "empty":
        return None

    type_config = map_params_to_type(federate_type)

    # Build Publications List
    publications = []
    pubs = node_data.get("publications", {})
    for key, unit in pubs.items():
        # Determine data type based on unit or key context (simplistic heuristic)
        # Using "string" (JSON) for control, "double" for physical values
        data_type = "string" if unit == "json" else "double"
        
        publications.append({
            "global": True,
            "key": key,
            "type": data_type,
            "unit": unit,
            "tags": {"logger": "yes"} # Default to logging everything for now
        })

    # Build Subscriptions List
    subscriptions = []
    subs = node_data.get("subscriptions", {})
    for key, unit in subs.items():
        data_type = "string" if unit == "json" else "double"
        subscriptions.append({
            "key": key,
            "type": data_type,
            "unit": unit
        })

    # Construct HELICS config
    helics_config = {
        "name": federate_id, # Using ID as the strict HELICS name to avoid collisions
        "log_level": "warning",
        "period": global_time_step,
        "terminate_on_error": True,
        # "broker_address": "udp://0.0.0.0", # Usually handled by runner/CST
        "publications": publications,
        "subscriptions": subscriptions,
        # Endpoints can be added here if used
    }

    # Construct Federate Container Config
    federate_config = {
        "logger": False,
        "image": type_config["image"],
        "command": type_config["command"],
        "federate_type": "value", # Defaulting to combo in CST terms
        "HELICS_config": helics_config
    }
    
    # Determine directory for configuration files
    # The runner expects configs to be available. 
    # For now, we embed specific configs into the 'command' or assume ENV vars?
    # The requirement says "runner.json files are derived from...". 
    # Usually, specific parameters (like grid file path) need to be passed to the federate.
    # CST usually looks for a config JSON. We might need to handle extra properties (like 'layout') here 
    # or ensure they are written to a file that the federate loads.
    # For this snippet, we focus on the runner structure requested.
    
    return federate_config

# --- Main Generation Loop ---

federation_config = {}
global_time_step = general_cfg.get("time_step", 1)

for node in tree.all_nodes():
    # Only process specific leaf nodes or major components that act as federates.
    # Based on the previous logic, 'grid', 'house', 'pv', 'battery', 'hems' seem to be the agents.
    # However, usually sub-components (pv inside house) might be inside the house federate
    # OR separate federates. The tree expansion suggests they are separate nodes. 
    # If every node in the tree is a HELICS federate:
    
    fed_conf = generate_federate_config(node, global_time_step)
    if fed_conf:
        # Use valid JSON key for the federate dictionary
        federation_config[node.identifier] = fed_conf

runner_json_structure = {
    "federation": federation_config
}

print(json.dumps(runner_json_structure, indent=2))

{
  "federation": {
    "mv-grid_0": {
      "logger": false,
      "image": "gridlock-grid:latest",
      "command": "python3 main.py",
      "federate_type": "value",
      "HELICS_config": {
        "name": "mv-grid_0",
        "log_level": "warning",
        "period": 300,
        "terminate_on_error": true,
        "publications": [
          {
            "global": true,
            "key": "mv-grid_0/bus_2/voltage",
            "type": "double",
            "unit": "V",
            "tags": {
              "logger": "yes"
            }
          },
          {
            "global": true,
            "key": "mv-grid_0/bus_3/voltage",
            "type": "double",
            "unit": "V",
            "tags": {
              "logger": "yes"
            }
          },
          {
            "global": true,
            "key": "mv-grid_0/bus_4/voltage",
            "type": "double",
            "unit": "V",
            "tags": {
              "logger": "yes"
            }
          },
